In [ ]:
import sys, os
import importlib
import torch.nn.functional as F
repo_start = f'../'
sys.path.append(repo_start)

from modules.utils.imports import *
from modules.binn_eql.model_wrapper_2d import model_wrapper
from modules.binn_eql.build_binn_eql_net import BINN
from modules.loaders.format_data import format_data_general
from modules.symbolic_net.visualize_surface import visualize_surface
from modules.generate_data.simulate_system import wave_pinning

In [ ]:
# load params from configuration file
dir_name = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql_net/runs/9_uniform_fc_init_one_dupe_l05/binn_eql_l05_reg_0.001_repeat_5'

config = {}
exec(Path(f'{dir_name}/config.cfg').read_text(encoding="utf8"), {}, config)

# Training data params
training_data_path = config['training_data_path']
species = int(config['species'])
dimensions = int(config['dimensions'])
epsilon = float(config['epsilon'])
points = int(config['points'])

# BINN params
diff_coeffs = [float(x) for x in config['diff_coeffs'].strip("()").split()]

# Symbolic Net params
duplicates = int(config['duplicates'])
degree = int(config['degree'])
l05_reg = float(config['l05_reg'])
param_bounds = float(config['param_bounds'])

In [ ]:
# Load model
binn = BINN(
    dimensions=dimensions,
    species=species, 
    duplicates=duplicates,
    diff_coeffs=diff_coeffs,
    degree=degree,
    l05_reg=l05_reg,
    param_bounds=param_bounds)

binn.to('cpu')

parameters = binn.parameters()

opt = torch.optim.Adam(parameters, lr=0.001)

model = model_wrapper(
    model=binn,
    optimizer=opt,
    loss=binn.loss,
    augmentation=None,
    save_name=f'{dir_name}/binn')

model.load(f"{dir_name}/binn_best_val_model", device='cpu')

# analyze_model(model, dir_name, training_data, 'cpu')

In [ ]:
# Load training data (columns: x*dimensions, t, species concentrations)
xtuv = format_data_general(2, 2, file=training_data_path)

# Create mesh of u and v values sampled during training
u_triangle_mesh, v_triangle_mesh = lltriangle(xtuv[:, -2], xtuv[:, -1])
u, v = np.ravel(u_triangle_mesh), np.ravel(v_triangle_mesh)
training_data_nans = np.stack((u, v), axis=1)

# Remove nans from training data
mask = ~np.isnan(training_data_nans).any(axis=1)
training_data = training_data_nans[mask]

# INVESTIGATE EQUATIONS/SURFACES

In [ ]:
model.model.remove_insignificant_terms(torch.tensor(training_data))
model.model.fix_cheating_hill_functions(torch.tensor(training_data))
model.model.generate_equation()

In [ ]:
i = 3
u, v = training_data_nans[:, 0], training_data_nans[:, 1]
F_true = (-0.003 * (1 - u**3.788 / (1 + 0.00035 * u**3.788))).reshape(501, 501)

hill_start = model.model.reaction.eql_layer.num_poly_features
weights = model.model.reaction.eql_layer.fc.weight[0][hill_start:]
features = model.model.reaction.eql_layer.hill(torch.tensor(training_data_nans))
# weights = model.model.unpack_coeffs()[0]
# features = model.model.reaction.eql_layer.poly(torch.tensor(training_data_nans).float())

F_eql = (weights * features).cpu().detach().numpy()

visualize_surface(dir_name, u_triangle_mesh, v_triangle_mesh, F_true, F_eql[:, i].reshape(501, 501))

In [ ]:
params = 1, 1, 0.01
# F_true = wave_pinning(training_data_nans, params).reshape(501, 501)
F_true = (4.09326 * u + 
           -3.76719 * v + 
           -0.10483 * u * u + 
           -2.96473 * u**1.36452 / (1 + 1.04535 * u**1.36452) + 
           -6.01141 * v * u**0.97592 / (1 + 0.00446 * u**0.97592) + 
           5.83893 * (1 - v**1.31269 / (1 + 0.10954 * v**1.31269)) +
           -1.81741 * u * (1 - v**0.00024 / (1 + 2.18992 * v**0.00024))
          ).reshape(501, 501)

F_eql = model.model.reaction(torch.tensor(training_data_nans).float()).reshape(501, 501).detach().numpy()

visualize_surface(dir_name, u_triangle_mesh, v_triangle_mesh, F_true, F_eql)

In [ ]:
params = 1, 1, 0.01
F_true = wave_pinning(training_data_nans, params).reshape(501, 501)
F_eql = model.model.reaction(torch.tensor(training_data_nans).float()).reshape(501, 501).detach().numpy()

visualize_surface(dir_name, u_triangle_mesh, v_triangle_mesh, F_true, F_eql)

# INVESTIGATE LOSS

In [ ]:
pred = model.model.surface_fitter(torch.from_numpy(xtuv[-100:, :3]).float())
true = torch.from_numpy(xtuv[-100:, -2:]).float()

In [ ]:
model.model.loss(pred, true)

In [ ]:
(-1.1873e-01 - -0.1087)**2 + (1.3732e-03 - 0.2862)**2